# Anatomy of a bilingual TTS corpus: what 73 minutes of *clean* speech actually looks like

Most speech datasets are scraped at scale and cleaned statistically. This one went the
other way: 555 clips, every single one listened to by a human, transcript corrected by
hand, acoustic quality gated before review. Small, but genuinely clean.

That makes it a rare chance to ask: **what does a properly curated speech corpus look
like from the inside?** In this notebook we take it apart layer by layer:

1. **Corpus shape** - balance, durations, and why the 4-18 s window exists
2. **Acoustic fingerprints** - SNR, loudness, spectral character of the two sources
3. **A map of the corpus** - every clip embedded via MFCCs and projected to 2D (interactive)
4. **Pitch and rhythm** - F0 distributions, pause structure, speaking rate
5. **The text side** - vocabulary, Zipf's law, disfluencies in real lecture speech
6. **ASR vs human** - word error rate implemented from scratch, plus the actual diffs
7. **Spectrograms and listening** - because plots only get you so far
8. **Using it** - a minimal PyTorch `Dataset` you can lift into a TTS training loop

Dataset: [Indian English + Hindi TTS Speech Dataset](https://www.kaggle.com/datasets/kavinthakurkm/indian-english-hindi-tts-dataset) -
full curation pipeline on [GitHub](https://github.com/auraCodesKM/sarvam-tts-dataset).

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
import IPython.display as ipd
from pathlib import Path

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})

manifest = next(Path("/kaggle/input").rglob("manifest.csv"))
DATA_DIR = manifest.parent
df = pd.read_csv(manifest)
df["wav"] = df["clip_id"].map(lambda c: str(DATA_DIR / "clips" / f"{c}.wav"))

C = {"en-IN": "#3d8fb8", "hi-IN": "#d98836"}  # one palette, used everywhere

print(f"{len(df)} clips, {df.duration_s.sum()/60:.1f} minutes")
df.head(3)

## 1. Corpus shape

Two languages, one speaker each, deliberately different registers:

- **English**: an NPTEL thermodynamics lecture - spontaneous classroom speech,
  disfluencies ("uh", "um") kept in the transcript on purpose
- **Hindi**: a studio narration of Premchand's essay *sahitya ka uddeshya* - scripted,
  read aloud, much more controlled delivery

Spontaneous vs. read speech is one of the classic contrasts in speech science, and
having both in one corpus (same format, same QC) is what makes the rest of this
notebook interesting.

In [ ]:
overview = df.groupby("language").agg(
    clips=("clip_id", "count"),
    minutes=("duration_s", lambda s: round(s.sum() / 60, 1)),
    mean_dur_s=("duration_s", "mean"),
    median_snr_db=("snr_db", "median"),
    mean_lufs=("lufs", "mean"),
).round(2)
display(overview)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for lang, sub in df.groupby("language"):
    axes[0].hist(sub["duration_s"], bins=28, alpha=0.65, label=lang, color=C[lang])
axes[0].set_xlabel("clip duration (s)"); axes[0].set_ylabel("clips")
axes[0].set_title("Durations sit inside the 4-18 s design window"); axes[0].legend()

# cumulative minutes: is either language front-loaded with long clips?
for lang, sub in df.groupby("language"):
    d = np.sort(sub["duration_s"].values)[::-1].cumsum() / 60
    axes[1].plot(np.arange(1, len(d) + 1), d, label=lang, color=C[lang])
axes[1].set_xlabel("clips (longest first)"); axes[1].set_ylabel("cumulative minutes")
axes[1].set_title("How the minutes accumulate"); axes[1].legend()
plt.tight_layout(); plt.show()

The window matters: clips under ~4 s carry too little prosodic context for TTS
training, and clips over ~18 s blow up attention-based alignment. Both sources ended
up with similar mass in the same band, which is what you want from a shared pipeline.

## 2. Acoustic fingerprints

`snr_db` and `lufs` come from the QC stage. But we can go further and measure the
*spectral* character of each recording chain ourselves. One pass over all 555 files;
we'll reuse these features for the corpus map in section 3.

In [ ]:
from tqdm.auto import tqdm

feats = []
for _, row in tqdm(df.iterrows(), total=len(df)):
    y, sr = librosa.load(row.wav, sr=None)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    cent = librosa.feature.spectral_centroid(y=y, sr=sr)
    # pause structure: fraction of the clip below -35 dB relative to peak
    intervals = librosa.effects.split(y, top_db=35)
    speech_s = sum((e - s) for s, e in intervals) / sr
    feats.append({
        "clip_id": row.clip_id,
        "mfcc_mean": mfcc.mean(axis=1),
        "mfcc_std": mfcc.std(axis=1),
        "centroid_hz": float(cent.mean()),
        "silence_ratio": 1 - speech_s / (len(y) / sr),
    })

F = pd.DataFrame(feats)
df = df.merge(F[["clip_id", "centroid_hz", "silence_ratio"]], on="clip_id")
X = np.stack([np.concatenate([f["mfcc_mean"], f["mfcc_std"]]) for f in feats])
print("feature matrix:", X.shape)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.6))

for lang, sub in df.groupby("language"):
    axes[0].scatter(sub["snr_db"], sub["lufs"], s=7, alpha=0.5, label=lang, color=C[lang])
axes[0].set_xlabel("SNR (dB)"); axes[0].set_ylabel("loudness (LUFS)")
axes[0].set_title("QC metrics: SNR vs loudness"); axes[0].legend()

for lang, sub in df.groupby("language"):
    axes[1].hist(sub["centroid_hz"], bins=30, alpha=0.65, label=lang, color=C[lang])
axes[1].set_xlabel("mean spectral centroid (Hz)")
axes[1].set_title("Spectral brightness"); axes[1].legend()

for lang, sub in df.groupby("language"):
    axes[2].hist(sub["silence_ratio"] * 100, bins=30, alpha=0.65, label=lang, color=C[lang])
axes[2].set_xlabel("% of clip that is silence/pause")
axes[2].set_title("Pause structure"); axes[2].legend()
plt.tight_layout(); plt.show()

Three separate signals, one story. The studio Hindi narration is quieter-floored
(higher SNR), and the lecture recording is brighter (higher spectral centroid - lecture
hall, distant mic, more high-frequency room response). The pause plot is my favorite:
spontaneous lecture speech pauses *more and more irregularly* than read narration,
because the speaker is thinking while talking. You can see cognition in a histogram.

## 3. A map of the corpus

Every clip is now a 40-dim vector (MFCC means + stds). Project to 2D and each point
becomes one clip - hover to read what's being said. If the curation is consistent,
we expect two tight islands (two speakers, two recording chains), not a smear.

In [ ]:
from sklearn.preprocessing import StandardScaler

Xs = StandardScaler().fit_transform(X)
try:
    from umap import UMAP
    emb = UMAP(n_neighbors=20, min_dist=0.15, random_state=42).fit_transform(Xs)
    method = "UMAP"
except Exception:
    from sklearn.manifold import TSNE
    emb = TSNE(perplexity=30, random_state=42).fit_transform(Xs)
    method = "t-SNE"

df["x"], df["y"] = emb[:, 0], emb[:, 1]

import plotly.express as px
fig = px.scatter(
    df, x="x", y="y", color="language",
    color_discrete_map=C,
    hover_data={"x": False, "y": False, "clip_id": True, "duration_s": ":.1f",
                "snr_db": True, "text": True},
    title=f"{method} projection of MFCC features - each point is one clip",
    opacity=0.75, height=520,
)
fig.update_traces(marker=dict(size=6))
fig.update_layout(template="plotly_white", xaxis_title=None, yaxis_title=None,
                  xaxis_showticklabels=False, yaxis_showticklabels=False)
fig.show()

Two clean islands, essentially zero cross-contamination - no mislabeled clips, no
stray second speaker, no clips with wildly different acoustics hiding in the set. This
plot is doing real QA work: on scraped corpora it's usually where the skeletons show up.

## 4. Pitch and rhythm

Fundamental frequency (F0) via YIN on a sample of clips per language. Together with
the pause analysis above, this characterizes the *prosody* a TTS model would learn
from each speaker.

In [ ]:
rng = np.random.default_rng(0)
sample = pd.concat([
    df[df.language == l].sample(min(60, (df.language == l).sum()), random_state=0)
    for l in ["en-IN", "hi-IN"]
])

f0_rows = []
for _, row in tqdm(sample.iterrows(), total=len(sample)):
    y, sr = librosa.load(row.wav, sr=None, duration=8.0)
    f0 = librosa.yin(y, fmin=60, fmax=350, sr=sr)
    f0 = f0[(f0 > 60) & (f0 < 350)]
    if len(f0):
        f0_rows.append({"language": row.language, "median_f0": float(np.median(f0)),
                        "f0_iqr": float(np.subtract(*np.percentile(f0, [75, 25])))})
f0df = pd.DataFrame(f0_rows)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for lang, sub in f0df.groupby("language"):
    axes[0].hist(sub["median_f0"], bins=24, alpha=0.65, label=lang, color=C[lang])
axes[0].set_xlabel("median F0 (Hz)"); axes[0].set_title("Pitch level"); axes[0].legend()
for lang, sub in f0df.groupby("language"):
    axes[1].hist(sub["f0_iqr"], bins=24, alpha=0.65, label=lang, color=C[lang])
axes[1].set_xlabel("F0 interquartile range (Hz)")
axes[1].set_title("Pitch variability (expressiveness proxy)"); axes[1].legend()
plt.tight_layout(); plt.show()

print(f0df.groupby("language").median().round(1))

## 5. The text side

TTS quality depends as much on transcript coverage as on audio. Three quick checks:
vocabulary growth, Zipf behavior, and - since the English source is a live lecture -
how much disfluency actually made it into the transcripts (spoiler: a lot, and that's
a feature: models trained on it learn realistic hesitation).

In [ ]:
import re
from collections import Counter

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))

for lang, sub in df.groupby("language"):
    words, seen, growth = [], set(), []
    for t in sub["text"]:
        for w in re.findall(r"[\w\u0900-\u097F']+", t.lower()):
            words.append(w); seen.add(w); growth.append(len(seen))
    axes[0].plot(growth, label=f"{lang} ({len(seen)} types / {len(words)} tokens)", color=C[lang])

    freqs = np.array(sorted(Counter(words).values(), reverse=True))
    axes[1].loglog(np.arange(1, len(freqs) + 1), freqs, label=lang, color=C[lang])

axes[0].set_xlabel("tokens seen"); axes[0].set_ylabel("unique words")
axes[0].set_title("Vocabulary growth"); axes[0].legend(fontsize=8)
axes[1].set_xlabel("rank"); axes[1].set_ylabel("frequency")
axes[1].set_title("Zipf check (log-log)"); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
en = df[df.language == "en-IN"]
fillers = ["uh", "um"]
counts = {f: en["text"].str.lower().str.count(rf"\b{f}\b").sum() for f in fillers}
per_min = {f: c / (en.duration_s.sum() / 60) for f, c in counts.items()}
print("Disfluencies kept in the English lecture transcripts:")
for f in fillers:
    print(f"  '{f}': {int(counts[f])} occurrences ({per_min[f]:.1f} per minute of speech)")
pct = (en["text"].str.lower().str.contains(r"\buh\b|\bum\b")).mean() * 100
print(f"  {pct:.0f}% of English clips contain at least one hesitation marker")

## 6. ASR vs human: measuring what the reviewer fixed

The manifest ships **both** the raw ASR output and the human-corrected transcript -
which quietly turns this TTS dataset into a small ASR benchmark. Word error rate via
dynamic programming, no dependencies:

In [ ]:
def wer(ref: str, hyp: str) -> float:
    r, h = ref.split(), hyp.split()
    d = np.zeros((len(r) + 1, len(h) + 1), dtype=int)
    d[:, 0], d[0, :] = np.arange(len(r) + 1), np.arange(len(h) + 1)
    for i in range(1, len(r) + 1):
        for j in range(1, len(h) + 1):
            d[i, j] = min(d[i-1, j] + 1, d[i, j-1] + 1,
                          d[i-1, j-1] + (r[i-1] != h[j-1]))
    return d[-1, -1] / max(len(r), 1)

df["wer"] = [wer(t, a) for t, a in zip(df["text"], df["asr_transcript"])]
display(df.groupby("language").agg(
    edited_pct=("edited", lambda s: round(100 * s.mean(), 1)),
    mean_wer=("wer", lambda s: round(s.mean(), 4)),
))

print("\nEvery correction the human reviewer actually made:\n")
for _, row in df[df.edited == True].iterrows():
    print(f"--- {row.clip_id}")
    print(f"ASR:   {row.asr_transcript}")
    print(f"human: {row.text}\n")

Near-zero WER is expected *by construction* (the human started from the ASR text),
so the honest reading is the edit rate: how often ASR was already shippable. For clean,
single-speaker audio in these two languages, the answer is: almost always. The
corrections that were needed are all real errors, not style tweaks.

## 7. Spectrograms, then ears

Pick the highest-SNR mid-length clip per language. Waveform, log-mel spectrogram, then
the audio itself - always listen before trusting a dataset.

In [ ]:
def show_clip(row):
    y, sr = librosa.load(row.wav, sr=None)
    fig, axes = plt.subplots(2, 1, figsize=(11, 4.8), sharex=True)
    librosa.display.waveshow(y, sr=sr, ax=axes[0], color=C[row.language])
    axes[0].set_title(f"{row.clip_id} | {row.duration_s:.1f}s | SNR {row.snr_db} dB")
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=80)
    librosa.display.specshow(librosa.power_to_db(S, ref=np.max), sr=sr,
                             x_axis="time", y_axis="mel", ax=axes[1], cmap="magma")
    axes[1].set_title("log-mel (80 bands)")
    plt.tight_layout(); plt.show()

for lang in ["en-IN", "hi-IN"]:
    sub = df[df.language == lang]
    row = sub[sub.duration_s.between(8, 12)].nlargest(1, "snr_db").iloc[0]
    show_clip(row)
    print(f"{lang}: {row.text}\n")
    display(ipd.Audio(row.wav))

## 8. Actually using it: a minimal PyTorch dataset

Everything above says the data is trainable. Here's the piece most EDA notebooks skip -
the bridge from manifest to training loop. This is deliberately minimal (no phonemizer,
no mel caching) but it's the real shape of a TTS data pipeline.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class TTSClips(Dataset):
    def __init__(self, frame, language=None):
        self.rows = frame[frame.language == language] if language else frame
        self.rows = self.rows.reset_index(drop=True)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        row = self.rows.iloc[i]
        y, sr = librosa.load(row.wav, sr=24000)
        mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=80)
        return {
            "text": row.text,
            "mel": torch.from_numpy(librosa.power_to_db(mel)).float(),
            "duration_s": row.duration_s,
        }

ds = TTSClips(df, language="hi-IN")
item = ds[0]
print(f"{len(ds)} Hindi clips | mel shape {tuple(item['mel'].shape)} | text: {item['text'][:60]}...")

## Wrap-up

What a curated corpus looks like from the inside:

- Two clean acoustic islands in embedding space - no label noise, no stray speakers
- High, tight SNR; loudness normalized; durations inside the TTS-friendly window
- Prosody differs *systematically* between spontaneous lecture and read narration -
  visible in pauses, pitch variability, and disfluency rate
- Raw ASR + corrected text in the same rows makes it a free ASR benchmark
- 555 clips is small for TTS from scratch, right-sized for fine-tuning one speaker

Things worth building on top of this:

- Fine-tune VITS / StyleTTS2 / Parler-TTS on the Hindi narrator (the cleaner speaker)
- Benchmark Whisper or an Indic ASR model against the human references
- Use the English disfluencies to study hesitation modeling in TTS

Pipeline code, QC report, and how every clip got here:
https://github.com/auraCodesKM/sarvam-tts-dataset

*If this was useful, an upvote on the notebook and the
[dataset](https://www.kaggle.com/datasets/kavinthakurkm/indian-english-hindi-tts-dataset)
helps other people working on Indian-language speech find it.*